In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!rm -rf colab_llm_utils

In [3]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1224, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1224 (delta 79), reused 93 (delta 43), pack-reused 1092 (from 1)
Receiving objects: 100% (1224/1224), 799.55 KiB | 19.99 MiB/s, done.
Resolving deltas: 100% (768/768), done.


In [4]:
%%capture
!pip install datasets transformers evaluate rouge_score accelerate
!pip install git+https://github.com/google-research/bleurt.git
!pip install --upgrade bitsandbytes

In [5]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [6]:
import colab_llm_utils
from importlib import reload
reload(colab_llm_utils)

<module 'colab_llm_utils' from '/content/colab_llm_utils/__init__.py'>

In [7]:
dir(colab_llm_utils.misc)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'biased_idxmax',
 'cap_interp_curve',
 'clip_series',
 'cuda_duignostics',
 'dist_mh',
 'dist_transform',
 'dist_transform2',
 'extra_cols',
 'gaussian_valley',
 'generalized_gaussian_valley',
 'is_number',
 'likelihood',
 'norm_series',
 'np',
 'sim_cosine']

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoConfig, BitsAndBytesConfig, GenerationConfig
from accelerate import infer_auto_device_map, init_empty_weights

In [9]:

from scipy import stats
import re
import sys
import gc
import os
import string
import math
from datetime import datetime

In [10]:
import torch
from torch.nn import functional as F


In [11]:


model_config = colab_llm_utils.configs.models.qwen3_8b
large_model_config = colab_llm_utils.configs.models.qwen3_8b
embedding_model_config = colab_llm_utils.configs.models.qwen3_8b
#model_config = colab_llm_utils.configs.models.t5_ba
#large_model_config = colab_llm_utils.configs.models.t5_xl
#embedding_model_config = colab_llm_utils.configs.models.t5_base

short_name = model_config['model_name'].split('/')[-1]
short_name_large = large_model_config['model_name'].split('/')[-1]
special_tag_2 = "rag"
special_tag_3 = 'thinking'

base_path = "/content/drive/MyDrive/phase3/Llama"
LARGE_SUFFIX = "_large"
FILL_LARGE = False

special_tag = ""


# Get the current datetime object

# Format the datetime object into a specific string format
formatted_date = datetime.now().strftime("%Y-%m-%d_%H:%M")

In [12]:
POS_TERMS = ["INTJ",
"VERB",
"PUNCT",
"SCONJ",
"PROPN",
"NUM",
"PRON",
"AUX",
"CCONJ",
"X",
"ADJ",
"NOUN",
"ADP",
"SPACE",
"ADV",
"PART"]

In [13]:
# filters were good on 3b chat xsum and cnn
filt_none = []
#filt_1a = ["SPACE", "PUNCT", "NUM", "PRON", "NOUN", "X", "ADP"]
#filt_1b = ["SPACE", "PUNCT", "NUM", "PRON", "AUX", "VERB", "PROPN", "INTJ"]
SEM_POS_FILTER = filt_none
LEX_POS_FILTER = filt_none
POS_POS_FILTER =filt_none
STRICTNESS = .3#.5
DISTANCE_LIMIT = 5

dataset_list = [#colab_llm_utils.configs.datasets.triviaqa,
                colab_llm_utils.configs.datasets.arc_challenge,
                #colab_llm_utils.configs.datasets.sciq,
                #colab_llm_utils.configs.datasets.nqopen,
                #colab_llm_utils.configs.datasets.hotpotqa,
                #colab_llm_utils.configs.datasets.cnn_dailymail,
                #colab_llm_utils.configs.datasets.wmt14de,
                #colab_llm_utils.configs.datasets.wmt19de,
                #colab_llm_utils.configs.datasets.wmt14
                #colab_llm_utils.configs.datasets.xsum,
                #colab_llm_utils.configs.datasets.samsum
                ]

dataframes = []
emb_dicts = []
auc_tables = []
all_results = []
for dataset in dataset_list:
  drive_path = os.path.join(base_path, dataset["clean_name"])
  results_path = os.path.join(base_path, "results", dataset['clean_name'])
  if not os.path.exists(results_path):
    os.makedirs(results_path)
  results_tag = f"MS:{short_name}{special_tag}ML{short_name_large}{formatted_date}"
  DICT_ANS = dataset['dict_ans']
  SELF_CONF = False
  PTRUE = False
  if PTRUE:
    short_name = short_name + "ptrue"

  if SELF_CONF:
    short_name = short_name + "conf"

  #TODO add metric to dataset config
  METRIC = 'f1'

  df1 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
  df2 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name_large}{special_tag_2}.pickle"))
  df3 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name_large}{special_tag_3}.pickle"))
  df1["correct"] = (df1[METRIC] > .01).astype(float)
  df2["correct"] = (df2[METRIC] > .01).astype(float)
  df3["correct"] = (df3[METRIC] > .01).astype(float)
  colab_llm_utils.misc.extra_cols(df1)
  colab_llm_utils.misc.extra_cols(df2)
  colab_llm_utils.misc.extra_cols(df3)


  print(f"Dataset: {dataset['clean_name']}")
  print(f"Model: {model_config['model_name']}")

  tokenizer = AutoTokenizer.from_pretrained(embedding_model_config['model_name'], token = HF_TOKEN)
  stopword_ids = set()

  embedding_layer = colab_llm_utils.model_tools.get_or_load_embedding(base_path, embedding_model_config, input_embeddings = True)
  embedder = colab_llm_utils.model_tools.tokenizer_embedder(embedding_layer, tokenizer)

  conf_metrics = ['chow_quantile', 'log_chow_av'] # baselines


  emb_dict, pos_dict = colab_llm_utils.confidence.get_embedding_pos_dicts(df1, embedder, tokenizer)
  emb_dicts.append(emb_dict)
  if PTRUE:
    conf_metrics = conf_metrics + ['p_true']

  DO_ABLATIONS = True
  if DO_ABLATIONS:
    colab_llm_utils.confidence.get_cs_thresh_likes(df1, emb_dict, pos_dict, tokenizer, position_correct = False, collapse_prefix = True, sim_thresh = 2, allow_empty = True, lex_pos_filt= LEX_POS_FILTER, bidir_col = False, sem_pos_filt=SEM_POS_FILTER, tag = 'just_collapse')
    colab_llm_utils.confidence.get_cs_thresh_likes(df1, emb_dict,  pos_dict, tokenizer, position_correct = True, collapse_prefix = False, distance_limit= DISTANCE_LIMIT, sim_thresh = 2, pos_pos_filt=POS_POS_FILTER, future_sim_thresh=1.6, tag = 'just_position')
    conf_metrics = conf_metrics + ['just_position_cs_log_chow_av', 'just_collapse_cs_log_chow_av']

  DO_THRESH = True
  if DO_THRESH:

    colab_llm_utils.confidence.get_cs_thresh_likes(df1, emb_dict,  pos_dict, tokenizer, position_correct = True, distance_limit=DISTANCE_LIMIT, collapse_prefix = True, sim_thresh = STRICTNESS, bidir_col = False, clip = 1,
                                                   allow_empty = True, lex_pos_filt= LEX_POS_FILTER, sem_pos_filt=SEM_POS_FILTER, pos_pos_filt=POS_POS_FILTER, future_sim_thresh=1.6, tag = 'full_thresh') #.4
    colab_llm_utils.confidence.get_cs_thresh_likes(df1, emb_dict,  pos_dict, tokenizer, position_correct = False, collapse_prefix = False, sim_thresh = STRICTNESS, allow_empty = True, lex_pos_filt= LEX_POS_FILTER, sem_pos_filt=SEM_POS_FILTER, tag = 'base_thresh')
    conf_metrics = conf_metrics + ['full_thresh_cs_log_chow_av',  'base_thresh_cs_log_chow_av']


  DO_CS = False
  if DO_CS:
    colab_llm_utils.confidence.get_cs_emb_likes(df1, emb_dict, tokenizer,  position_correct = False,  distance_limit=3, collapse_prefix = False, sim_adjust = 1, tag = 'base')
    colab_llm_utils.confidence.get_cs_emb_likes(df1, emb_dict, tokenizer,  position_correct = True, distance_limit=3,  collapse_prefix = True, sim_adjust = .5, tag = 'full_5')
    colab_llm_utils.confidence.get_cs_emb_likes(df1, emb_dict, tokenizer, position_correct = True, distance_limit=3, collapse_prefix = True, sim_adjust = 1, tag = 'full')
    conf_metrics = conf_metrics + ['base_cs_log_chow_av', 'full_cs_log_chow_av', 'full_5_cs_log_chow_av']


  if SELF_CONF:
    conf_metrics = conf_metrics + ['self_conf']
  #conf_metrics = ['chow_av','log_chow_av'] + list(df.columns[-80:])
  conf_metrics = list(set(conf_metrics))


  tab, quant_cols = colab_llm_utils.likelihood.quantile_analysis(df1, 'full_thresh_cs_likes', METRIC)
  tab2, quant_cols2 =  colab_llm_utils.likelihood.quantile_analysis(df1, 'likes', METRIC)

  df1['lex_frags'] = [m['lex_fragments'] for m in df1['just_collapsemetadata']]
  df1['lex_weight'] = [m['lex_frag_weight'] for m in df1['just_collapsemetadata']]
  df1['sem_frags'] = [m['semantic_collapses'] for m in df1['base_threshmetadata']]
  df1['sem_weight'] = [m['semantic_collapse_weight'] for m in df1['base_threshmetadata']]
  df1['position_adj'] = [m['position_adjustments'] for m in df1['just_positionmetadata']]
  df1['position_adj_weight'] = [m['position_adjust_weight'] for m in df1['just_positionmetadata']]
  df1['pos_list'] =[m['pos_list'] for m in df1['base_threshmetadata']]
  df1['change_list'] =[m['change_list'] for m in df1['base_threshmetadata']]
  df1['pos_list_lex'] =[m['pos_list'] for m in df1['just_collapsemetadata']]
  df1['change_list_lex'] =[m['change_list'] for m in df1['just_collapsemetadata']]
  df1['pos_list_pos'] =[m['pos_list'] for m in df1['just_positionmetadata']]
  df1['change_list_pos'] =[m['change_list'] for m in df1['just_positionmetadata']]
  df1['pos_list_pos2'] =[m['pos_col_pos_list'] for m in df1['just_positionmetadata']]
  df1['change_list_pos2'] =[m['pos_col_w_list'] for m in df1['just_positionmetadata']]
  df1['lex_frags_per_tok'] =  df1['lex_frags'] / df1['output_len']
  df1['lex_weight_per_tok'] =  df1['lex_weight'] / df1['output_len']
  df1['sem_frags_per_tok'] =  df1['sem_frags'] / df1['output_len']
  df1['sem_weight_per_tok'] =  df1['sem_weight'] / df1['output_len']
  df1['position_adj_per_tok'] =  df1['position_adj'] / df1['output_len']
  df1['position_adj_weight_per_tok'] =  df1['position_adj_weight'] / df1['output_len']



  df1['mean_prob'] = colab_llm_utils.misc.norm_series(df1['chow_av'], invert=True)
  df1['normed_log_chow_av'] = colab_llm_utils.misc.norm_series(df1['log_chow_av'], invert = True)

  df1['normed_chow_quantile'] = colab_llm_utils.misc.norm_series(df1['chow_quantile'], invert = True)


  print(short_name)
  print(dataset['clean_name'])
  cor_table = df1[conf_metrics + [METRIC]].corr(numeric_only=True)[METRIC].abs().sort_values(ascending=False)
  cor_table.to_csv(os.path.join(results_path, results_tag + "correlation1.csv"), index=True, header=True, float_format="%.6f")
  with open(os.path.join(results_path, results_tag + "correlation1.txt"), 'w') as f:
      # Convert dataframe to latex string and write to file
      f.write(cor_table.to_latex(index=True,
                                    caption='Correlation Table',
                                    label='tab:results',
                                    column_format='lrc'))
  cor_table


  DELEGATE_COLUMNS = ["random", 'normed_chow_quantile', 'normed_log_chow_av']
  CLEAN_NAME_DICT ={"random":'Random',
                    "normed_chow_quantile":"Quantile (.5)",
                    "normed_log_chow_av":"Chow Av.",
                    "n_cvar":"CVaR",
                    "collapse": "Lex. Collapse",
                    "t_embed": "Sem. Collapse",
                    "embed": "Sem. Dist",
                    "position": "Pos. Collapse",
                    "t_full": "Full",
                    "p_true_n": "P(True)",
                    "full": "Dist Full",
                    "post_hoc": "Post-hoc Model",
                    "post_hoc2": "Post-hoc Baselines",
                    "cvar": "CVaR",
                    "full_cvar": "Full Dist. CVaR",
                    "t_full_cvar": "Full CVaR",
                    "prod_av": "prod av",
                    "post_hoc_lm":"PH LM"}

  METRIC_DICT = {"f1": "F1 Score", "gpt_score": "GPT Score", "gpt_score_cont": "GPT Score", "bem": "BEM", "em": "EM", "bleurt": "BLEURT", "meteor": "METEOR", "rouge": "Rouge-L"}


  if DO_ABLATIONS:
    df1['collapse'] = colab_llm_utils.misc.norm_series(df1['just_collapse_cs_log_chow_av'], invert=True)
    df1['position'] = colab_llm_utils.misc.norm_series(df1['just_position_cs_log_chow_av'], invert=True)
    DELEGATE_COLUMNS = DELEGATE_COLUMNS + ['collapse', 'position',]


  if DO_THRESH:
    df1['t_full'] = colab_llm_utils.misc.norm_series(df1['full_thresh_cs_log_chow_av'], invert=True)
    df1['t_full_cvar'] = colab_llm_utils.misc.norm_series(df1['full_thresh_cs_cvar'], invert=True)
    df1['t_embed'] =  colab_llm_utils.misc.norm_series(df1['base_thresh_cs_log_chow_av'], invert=True)
    DELEGATE_COLUMNS = DELEGATE_COLUMNS + ['t_embed', 't_full']


  DELEGATE_COLUMNS = DELEGATE_COLUMNS + ['post_hoc_lm']

Dataset: ARC-Challenge
Model: Qwen/Qwen3-8B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Detected Vocab Size: 151936
Detected Embedding Dim: 4096
Qwen3-8B
ARC-Challenge


In [14]:
df1['responses'] = [r[0] for r in df1['responses']]

In [15]:
DELEGATE_COLUMNS

['random',
 'normed_chow_quantile',
 'normed_log_chow_av',
 'collapse',
 'position',
 't_embed',
 't_full',
 'post_hoc_lm']

In [18]:
device = "cpu"

# Single-Dim Knowledge

In [ ]:
for dataset in dataset_list:

 ######################################################### CASCADES PART

  casc = colab_llm_utils.cascades.MultiaxialCascade(df1, ["knowledge"], metric_col="correct")
  casc.register_axis_data(df2, (1,), 1.5)

  casc.normalize_dfs()
  casc.compute_cv_splits()


  casc.fit_post_hoc_lm_at(casc.origin, "prompts", "responses", num_epochs=5, device=device)
  #casc.set_pref_deferral_at = types.MethodType(set_pref_deferral_at, casc)
  #casc.set_pref_deferral_at(casc.origin, "def_axis", offset=-1)

  results = {}
  for d in DELEGATE_COLUMNS + ["post_hoc_lm"]:
    results[d] = casc.full_threshold_sim_temp(d, pref_def_column="preferred_deferral_lm")
  all_results.append(results)


  many_colors_husl = sns.color_palette("deep", len(DELEGATE_COLUMNS))


  lines = DELEGATE_COLUMNS

  f, axes = plt.subplots(1,2, figsize=(11,5))
  for (i, l) in enumerate(lines):
    axes[0].plot(results[l]['p_deferred'], results[l]['accs'], c=many_colors_husl[i])
  #plt.axhline(y=df_full['Correct_70b'].mean(), ls='--', c='red')
  #plt.plot(x_oracle, y_oracle, c='black')
  plt.legend([CLEAN_NAME_DICT[d] for d in DELEGATE_COLUMNS])
  #plt.title("Accuracy Gains vs Delegation")  # Set the title
  axes[0].set_xlabel("Deferral Rate")
  axes[0].set_ylabel(METRIC_DICT[METRIC])
  plt.savefig(os.path.join(results_path, results_tag + "deferral_curves.png"))

  colab_llm_utils.plotting.heatmap_results(results, label_override_dict=CLEAN_NAME_DICT)
  plt.savefig(os.path.join(results_path, results_tag + "heatmap1.png"))

  cal_results = {}
  for c in DELEGATE_COLUMNS:
    if c == 'random':
      continue

    cal_results[CLEAN_NAME_DICT[c]] = colab_llm_utils.metrics.confidence_metrics(df1, 'correct', c, invert=True)
    cal_results[CLEAN_NAME_DICT[c]]['kendall_tau'] = stats.kendalltau(df1[c], df1['correct']).statistic *-1 #invert tau
    cal_results[CLEAN_NAME_DICT[c]]['audc'] = results[c]['auc']
    cal_results[CLEAN_NAME_DICT[c]]['audc_40'] = results[c]['auc_40']

  auc_table = pd.DataFrame.from_dict(cal_results, orient='index').sort_values('audc', ascending=False)
  auc_table.to_csv(os.path.join(results_path, results_tag + "auc_table.csv"), index=True, header=True, float_format="%.6f")
  with open(os.path.join(results_path, results_tag + "auc_table.txt"), 'w') as f:
      # Convert dataframe to latex string and write to file
      f.write(auc_table.to_latex(index=True,
                                    caption='Calibration Table',
                                    label='tab:results',
                                    column_format='lrc'))
  auc_table
  auc_tables.append(auc_table)
  print(auc_table)

Registered ['knowledge: 1'] | Shape: (2000, 29) | Cost: 1.5
target dict shape (2000, 2)
targets: count    2000.000000
mean        0.155000
std         0.361995
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
dtype: float64
Number of classes: 2
Device: cpu
Training fold 1/4


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Single-Dim Reasoning

In [ ]:
for dataset in dataset_list:

 ######################################################### CASCADES PART

  casc = colab_llm_utils.cascades.MultiaxialCascade(df1, ["reasoning"], metric_col="correct")
  casc.register_axis_data(df3, (1,), 1.5)

  casc.normalize_dfs()
  casc.compute_cv_splits()


  casc.fit_post_hoc_lm_at(casc.origin, "prompts", "responses", num_epochs=6, device=device)
  #casc.set_pref_deferral_at = types.MethodType(set_pref_deferral_at, casc)
  #casc.set_pref_deferral_at(casc.origin, "def_axis", offset=-1)

  results = {}
  for d in DELEGATE_COLUMNS + ["post_hoc_lm"]:
    results[d] = casc.full_threshold_sim_temp(d, pref_def_column='preferred_deferral_lm')
  all_results.append(results)


  many_colors_husl = sns.color_palette("deep", len(DELEGATE_COLUMNS))


  lines = DELEGATE_COLUMNS

  f, axes = plt.subplots(1,2, figsize=(11,5))
  for (i, l) in enumerate(lines):
    axes[0].plot(results[l]['p_deferred'], results[l]['accs'], c=many_colors_husl[i])
  #plt.axhline(y=df_full['Correct_70b'].mean(), ls='--', c='red')
  #plt.plot(x_oracle, y_oracle, c='black')
  plt.legend([CLEAN_NAME_DICT[d] for d in DELEGATE_COLUMNS])
  #plt.title("Accuracy Gains vs Delegation")  # Set the title
  axes[0].set_xlabel("Deferral Rate")
  axes[0].set_ylabel(METRIC_DICT[METRIC])
  plt.savefig(os.path.join(results_path, results_tag + "deferral_curves.png"))

  colab_llm_utils.plotting.heatmap_results(results, label_override_dict=CLEAN_NAME_DICT)
  plt.savefig(os.path.join(results_path, results_tag + "heatmap1.png"))

  cal_results = {}
  for c in DELEGATE_COLUMNS:
    if c == 'random':
      continue

    cal_results[CLEAN_NAME_DICT[c]] = colab_llm_utils.metrics.confidence_metrics(df1, 'correct', c, invert=True)
    cal_results[CLEAN_NAME_DICT[c]]['kendall_tau'] = stats.kendalltau(df1[c], df1['correct']).statistic *-1 #invert tau
    cal_results[CLEAN_NAME_DICT[c]]['audc'] = results[c]['auc']
    cal_results[CLEAN_NAME_DICT[c]]['audc_40'] = results[c]['auc_40']

  auc_table = pd.DataFrame.from_dict(cal_results, orient='index').sort_values('audc', ascending=False)
  auc_table.to_csv(os.path.join(results_path, results_tag + "auc_table.csv"), index=True, header=True, float_format="%.6f")
  with open(os.path.join(results_path, results_tag + "auc_table.txt"), 'w') as f:
      # Convert dataframe to latex string and write to file
      f.write(auc_table.to_latex(index=True,
                                    caption='Calibration Table',
                                    label='tab:results',
                                    column_format='lrc'))
  auc_table
  auc_tables.append(auc_table)
  print(auc_table)

# Two-Way Cascade

In [ ]:
for dataset in dataset_list:

 ######################################################### CASCADES PART

  casc = colab_llm_utils.cascades.MultiaxialCascade(df1, ["knowledge", "reasoning"], metric_col="correct")
  casc.register_axis_data(df2, (1,0,), 1.5)
  casc.register_axis_data(df3, (0,1,), 3)

  casc.normalize_dfs()
  casc.compute_cv_splits()

  #casc.fit_post_hoc_at = types.MethodType(fit_post_hoc_at, casc)
  casc.fit_post_hoc_lm_at(casc.origin, "prompts", "responses", num_epochs=5, device=device)
  #casc.set_pref_deferral_at = types.MethodType(set_pref_deferral_at, casc)
  #casc.set_pref_deferral_at(casc.origin, "def_axis", offset=-1)

  results = {}
  for d in DELEGATE_COLUMNS + ["post_hoc_lm"]:
    results[d] = casc.full_threshold_sim_temp(d, pref_def_column='preferred_deferral_lm')
  all_results.append(results)


  many_colors_husl = sns.color_palette("deep", len(DELEGATE_COLUMNS))


  lines = DELEGATE_COLUMNS

  f, axes = plt.subplots(1,2, figsize=(11,5))
  for (i, l) in enumerate(lines):
    axes[0].plot(results[l]['p_deferred'], results[l]['accs'], c=many_colors_husl[i])
  #plt.axhline(y=df_full['Correct_70b'].mean(), ls='--', c='red')
  #plt.plot(x_oracle, y_oracle, c='black')
  plt.legend([CLEAN_NAME_DICT[d] for d in DELEGATE_COLUMNS])
  #plt.title("Accuracy Gains vs Delegation")  # Set the title
  axes[0].set_xlabel("Deferral Rate")
  axes[0].set_ylabel(METRIC_DICT[METRIC])
  plt.savefig(os.path.join(results_path, results_tag + "deferral_curves.png"))

  colab_llm_utils.plotting.heatmap_results(results, label_override_dict=CLEAN_NAME_DICT)
  plt.savefig(os.path.join(results_path, results_tag + "heatmap1.png"))

  cal_results = {}
  for c in DELEGATE_COLUMNS:
    if c == 'random':
      continue

    cal_results[CLEAN_NAME_DICT[c]] = colab_llm_utils.metrics.confidence_metrics(df1, 'correct', c, invert=True)
    cal_results[CLEAN_NAME_DICT[c]]['kendall_tau'] = stats.kendalltau(df1[c], df1['correct']).statistic *-1 #invert tau
    cal_results[CLEAN_NAME_DICT[c]]['audc'] = results[c]['auc']
    cal_results[CLEAN_NAME_DICT[c]]['audc_40'] = results[c]['auc_40']

  auc_table = pd.DataFrame.from_dict(cal_results, orient='index').sort_values('audc', ascending=False)
  auc_table.to_csv(os.path.join(results_path, results_tag + "auc_table.csv"), index=True, header=True, float_format="%.6f")
  with open(os.path.join(results_path, results_tag + "auc_table.txt"), 'w') as f:
      # Convert dataframe to latex string and write to file
      f.write(auc_table.to_latex(index=True,
                                    caption='Calibration Table',
                                    label='tab:results',
                                    column_format='lrc'))
  auc_table
  auc_tables.append(auc_table)
  print(auc_table)

In [ ]:
df1

In [ ]:
casc

In [ ]:
df1['preferred_deferral'].value_counts()

# Some Subset Analysis

In [ ]:
print(f"Baseline acc {df1['correct'].mean()}")

In [ ]:

print(f"base DF, requested escalation knowledge: {df1[df1["def_axis"] == 1]['correct'].mean()}")
print(f"base DF, requested escalation reasoning: {df1[df1["def_axis"] == 2]['correct'].mean()}")

In [ ]:
print(f"RAG DF, requested escalation knowledge: {df2[df1["def_axis"] == 1]['correct'].mean()}")
print(f"RAG DF, requested escalation reasoning: {df2[df1["def_axis"] == 2]['correct'].mean()}")

In [ ]:
print(f"Thinking DF, requested escalation knowledge: {df3[df1["def_axis"] == 1]['correct'].mean()}")
print(f"Thinking DF, requested escalation reasoning: {df3[df1["def_axis"] == 2]['correct'].mean()}")

## Items where RAG correct and thinking incorrect

In [ ]:
!pip install upsetplot

In [ ]:
ven_df = pd.DataFrame({"base": df1['correct'].astype(bool), "rag": df2['correct'].astype(bool), "think": df3['correct'].astype(bool)})

In [ ]:
from upsetplot import UpSet, from_indicators
import matplotlib.pyplot as plt

data = from_indicators(
    ['base', 'rag', 'think'],
    ven_df[['base', 'rag', 'think']].astype(bool)
)

UpSet(data, show_counts=True).plot()
plt.show()

## Preferred Deferral where RAG CORRECT THINK INCORRECT

In [ ]:

df1[((df2['correct'] == 1) & (df3['correct']== 0))]['def_axis'].value_counts()

In [ ]:
[print(f) for f in df2[((df2['correct'] == 1) & (df3['correct']== 0))]['prompts'].str[140:]]

## Preferred Deferral where RAG incorrect THINK CORRECT

In [ ]:
df1[((df2['correct'] == 0) & (df3['correct']== 1))]['def_axis'].value_counts()

In [ ]:
[print(f) for f in df1[((df2['correct'] == 0) & (df3['correct']== 1))]['prompts'].str[148:]]

In [ ]:

((df2['correct'] == 0) & (df3['correct']== 1)).sum()

In [ ]:
df3[df1["def_axis"] == 2]['correct'].mean()

In [ ]:
dir(casc)

In [ ]:
f, ax = plt.subplots(1,1, figsize=(11,5))
for i, res in enumerate(all_results[-3:]):
  ax.plot(res['post_hoc_lm']['p_deferred'], res['post_hoc_lm']['accs'], c=many_colors_husl[i])
  #plt.axhline(y=df_full['Correct_70b'].mean(), ls='--', c='red')
  #plt.plot(x_oracle, y_oracle, c='black')
  plt.legend(['Knowledge', 'Reasoning', 'Multi-Axial'])
  #plt.title("Accuracy Gains vs Delegation")  # Set the title
  ax.set_xlabel("Deferral Rate")
  ax.set_ylabel(METRIC_DICT[METRIC])


In [ ]:
import types
import copy

In [ ]:
def biased_idxmax(row, noise_scale=1e-8, bias=1e-1):
    values = row.values.astype(float)

    noise = np.random.normal(0, noise_scale, size=len(values))

    # hard bias: column 0 always gets a tiny boost
    values = values + noise
    values[0] += bias

    return row.index[np.argmax(values)]

In [ ]:
def fit_post_hoc_at(self,
    position,
    feature_cols,
    rf_kwargs=None,
    model_type = RandomForestClassifier
):
        """
        Returns out-of-fold predictions for each row in df using 5-fold CV.
        """
        df = self.registry[position]

        for c in feature_cols:
            if c not in list(df.columns):
                print(f"feature {c} not found in dataframe.")
                feature_cols.remove(c)

        if rf_kwargs is None:
            rf_kwargs = {}

        X = df[feature_cols].values

        target_dict = {0:df[self.metric_col]}
        # setup targets
        deferral_options = {0:position} # 0 index is keep for this model
        for i, _ in enumerate(self.axes_names):
            pos = copy.deepcopy(position)
            pos = pos[:i] + (pos[i] + 1,) + pos[i+1:]

            #pos[i] = pos[i] + 1
            deferral_options[i+1] = pos
            target_dict[i+1] = self.registry[pos][self.metric_col]

        target_df = pd.DataFrame(target_dict)

        print(f"target dict shape {target_df.shape}")
        #targets = target_df.idxmax(axis=1)
        targets = target_df.apply(biased_idxmax, axis=1)
        #targets = (target_df + np.random.normal(0, 1e-8, target_df.shape)).idxmax(axis=1)
        print(f"targets:{targets.describe()}")
        print(f"targets:{targets.value_counts()}")
        y = targets

        all_classes = np.arange(len(self.axes_names) + 1)
        n_classes = len(all_classes)
        oof_preds = np.zeros((len(df), n_classes))


        for i in range(self.kf.get_n_splits()):
            train_idx = df['fold'] != i
            val_idx = df['fold'] == i
            X_train, X_val = X[train_idx], X[val_idx]
            y_train = y[train_idx]

            model = model_type(
                **rf_kwargs
            )

            model.fit(X_train, y_train)
            probs = model.predict_proba(X_val)
            present_classes = model.classes_

            aligned = np.zeros((len(X_val), n_classes))
            aligned[:, present_classes] = probs

            oof_preds[val_idx] = aligned
        df['post_hoc'] = oof_preds[:,0]
        df['post_hoc'] = 1 - df['post_hoc']
        def_destinations = []
        for idx in oof_preds[:, 1:].argmax(axis=1):
          l = list(position)
          l[idx] = l[idx] + 1
          def_destinations.append(tuple(l))
        df['preferred_deferral']  = def_destinations
        return pd.DataFrame(oof_preds, index=df.index)



In [ ]:
    def set_pref_deferral_at(self, position, column, offset=-1):
        def_destinations = []
        for ax_idx in self.registry[position][column] + offset:
            l = list(position)
            l[ax_idx] = l[ax_idx] + 1
            def_destinations.append(tuple(l))
        self.registry[position]["preferred_deferral"] = def_destinations

In [ ]:
import types
import copy
casc.fit_post_hoc_at = types.MethodType(fit_post_hoc_at, casc)
#casc.resolve_full_deferred  = types.MethodType(resolve_full_deferred, casc)
#casc.full_threshold_sim_temp  = types.MethodType(full_threshold_sim_temp, casc)

In [ ]:
from colab_llm_utils import misc

In [ ]:
casc.fit_post_hoc_at((0,0,), ['chow_av', 'collapse', 'position', 't_embed', 't_full'] + quant_cols + quant_cols2)
#casc.full_threshold_sim_temp("oof_preds")

In [ ]:
np.zeros((10,10,2))[:,:,1]

In [ ]:
casc.registry[(0,0,)]

In [ ]:
df3 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name_large}{"thinking"}.pickle"))

In [ ]:
df3['correct'] = df3['f1'] > .01

In [ ]:
#df = df[df['word_len'] > 2]
#df_def = df_def[df_def['word_len']> 2]

In [ ]:
##colab_llm_utils.confidence.get_cs_thresh_likes(df, emb_dict,  pos_dict, tokenizer, position_correct = True, distance_limit=DISTANCE_LIMIT, collapse_prefix = True, sim_thresh = STRICTNESS, clip = 1,
#                                                   allow_empty = True, lex_pos_filt= ["PUNCT", "SPACE", "NUM", "X"], sem_pos_filt=["SPACE", "PUNCT", "NUM"], pos_pos_filt=["NUM"], future_sim_thresh=1.1, tag = 'full_thresh')

In [ ]:
#colab_llm_utils.confidence.get_cs_thresh_likes(df, emb_dict, pos_dict, tokenizer, position_correct = False, collapse_prefix = True, sim_thresh = 2, allow_empty = True, lex_pos_filt= ["PUNCT", "SPACE", "NUM", "X"], sem_pos_filt=SEM_POS_FILTER, tag = 'just_collapse')

In [ ]:
METRIC = 'gpt_score'
cor_table = df[conf_metrics + [METRIC]].corr(numeric_only=True)[METRIC].abs().sort_values(ascending=False)
METRIC = 'gpt_score'
cor_table

In [ ]:
reorder_methods =  ['Quantile (.5)',
 'Chow Av.',
 'Sem. Collapse',
 'Lex. Collapse',
 'Pos. Collapse',
 'Full']

In [ ]:
for i, t, in enumerate(auc_tables):
  print(dataset_list[i])
  print("\n")
  print(t.reindex(reorder_methods)[['auc', 'kendall_tau', 'brier']].to_latex())
  print("\n")

In [ ]:
audc_table = {}
for i, a in enumerate(auc_tables):
  audc_table[dataset_list[i]['clean_name']] = a['audc']

df_audc = pd.DataFrame(audc_table)

In [ ]:
reorder_methods =  ['Quantile (.5)',
 'Chow Av.',
 'Sem. Collapse',
 'Lex. Collapse',
 'Pos. Collapse',
 'Full',
 'Post-hoc Baselines',
 'Post-hoc Model']


In [ ]:
print(df_audc.reindex(reorder_methods).to_latex())

In [ ]:
def bold_max(s):
    '''
    Highlights the maximum value in a Series in bold.
    '''
    is_max = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_max]

In [ ]:
df_audc.reindex(reorder_methods)

In [ ]:
df_audc.reindex(reorder_methods).style.apply(bold_max, axis=0)

In [ ]:
DELEGATE_COLUMNS

In [ ]:
# FOR MAXPROB
pps = []
for delegation_results in all_results:
  parity_points = [ .2, .4]
  acc_points = []
  matched_at = []
  for p in parity_points:
    p_del = delegation_results['normed_log_chow_av']['p_deferred']
    acc_del = delegation_results['normed_log_chow_av']['accs']
    acc_points.append(np.interp(x=p, xp=p_del, fp=acc_del))
    print(acc_points)
    matched_at.append(pd.Series(delegation_results['t_full']['p_deferred'])[delegation_results['t_full']['accs'] > acc_points[-1]].reset_index(drop=True)[0])
    print(f"parity with Chow Av. score at {p} delegation of {acc_points[-1]:.4f} % achieved at {matched_at[-1]:.4f} delegation in Full")
  df_pp = pd.DataFrame({"Delegation Rate (MaxProb)":parity_points, "Accuracy":acc_points, "Surpassed At (Bi-Directional)":matched_at})
  df_pp['Average Cost (MaxProb)'] = 8 + df_pp['Delegation Rate (MaxProb)']*70
  df_pp['Average Cost (Bi-Directional)'] = 8 + df_pp['Surpassed At (Bi-Directional)']*70
  df_pp['Cost Reduction (%)'] =  (1-(df_pp['Average Cost (Bi-Directional)']/df_pp['Average Cost (MaxProb)']))*100
  #df_pp.to_csv(os.path.join(results_path, f"def_parity_{short_name}_{dataset['dataset_name']}.csv"))
  pps.append(df_pp)
  #

In [ ]:
for pp in pps:
  display(pp)

In [ ]:
df_audc.to_latex()

In [ ]:

DO_POST_HOC = True
if DO_POST_HOC:
    df_def['post_hoc'] = colab_llm_utils.cascades.post_hoc_oof(df_def, ['chow_av', 'collapse', 'position', 't_embed', 't_full'] + quant_cols + quant_cols2, 'd_target', random_state =1234)
    df_def['post_hoc'] = colab_llm_utils.misc.norm_series(df_def['post_hoc'], invert=True)
    DELEGATE_COLUMNS.append('post_hoc')
    df_def['post_hoc2'] = colab_llm_utils.cascades.post_hoc_oof(df_def, ['chow_av'] + quant_cols2, 'd_target', random_state =1234)
    df_def['post_hoc2'] = colab_llm_utils.misc.norm_series(df_def['post_hoc2'], invert=True)
    DELEGATE_COLUMNS.append('post_hoc2')

In [ ]:

results = {}
for d in DELEGATE_COLUMNS:
  results[d] = colab_llm_utils.cascades.cascade_scored_samples(df_def, d, METRIC, ml_suffix=LARGE_SUFFIX )
all_results.append(results)

results2 = {}
for d in DELEGATE_COLUMNS:
  results2[d] = colab_llm_utils.cascades.cascade_scored_samples(df_def, d, METRIC2, ml_suffix=LARGE_SUFFIX )

many_colors_husl = sns.color_palette("deep", len(DELEGATE_COLUMNS))


lines = DELEGATE_COLUMNS

f, axes = plt.subplots(1,2, figsize=(11,5))
for (i, l) in enumerate(lines):
  axes[0].plot(results[l]['p_deferred'], results[l]['accs'], c=many_colors_husl[i])
  axes[1].plot(results2[l]['p_deferred'], results2[l]['accs'], c=many_colors_husl[i])
#plt.axhline(y=df_full['Correct_70b'].mean(), ls='--', c='red')
#plt.plot(x_oracle, y_oracle, c='black')
plt.legend([CLEAN_NAME_DICT[d] for d in DELEGATE_COLUMNS])
#plt.title("Accuracy Gains vs Delegation")  # Set the title
axes[0].set_xlabel("Deferral Rate")
axes[1].set_xlabel("Deferral Rate")
axes[0].set_ylabel(METRIC_DICT[METRIC])
axes[1].set_ylabel(METRIC_DICT[METRIC2])
plt.savefig(os.path.join(results_path, results_tag + "deferral_curves.png"))

colab_llm_utils.plotting.heatmap_results(results, label_override_dict=CLEAN_NAME_DICT)
plt.savefig(os.path.join(results_path, results_tag + "heatmap1.png"))


colab_llm_utils.plotting.heatmap_results(results2, label_override_dict=CLEAN_NAME_DICT)
plt.savefig(os.path.join(results_path, results_tag + "heatmap2.png"))

cal_results = {}
for c in DELEGATE_COLUMNS:
  if c == 'random':
    continue

  cal_results[CLEAN_NAME_DICT[c]] = colab_llm_utils.metrics.confidence_metrics(df_def, 'correct', c, invert=True)
  cal_results[CLEAN_NAME_DICT[c]]['kendall_tau'] = stats.kendalltau(df_def[c], df_def['correct']).statistic *-1 #invert tau
  cal_results[CLEAN_NAME_DICT[c]]['audc'] = results[c]['auc']
  cal_results[CLEAN_NAME_DICT[c]]['audc_40'] = results[c]['auc_40']

auc_table = pd.DataFrame.from_dict(cal_results, orient='index').sort_values('audc', ascending=False)
auc_table.to_csv(os.path.join(results_path, results_tag + "auc_table.csv"), index=True, header=True, float_format="%.6f")
with open(os.path.join(results_path, results_tag + "auc_table.txt"), 'w') as f:
    # Convert dataframe to latex string and write to file
    f.write(auc_table.to_latex(index=True,
                                  caption='Calibration Table',
                                  label='tab:results',
                                  column_format='lrc'))
auc_table
#auc_tables.append(auc_table)
print(auc_table)

In [ ]:
df = dataframes[-1]


In [ ]:

all_types = set()
for p_l in df['pos_list']:
  for p in p_l:
    all_types.add(p)
for p_l in df['pos_list_lex']:
  for p in p_l:
    all_types.add(p)
for p_l in df['pos_list_pos2']:
  for p in p_l:
    all_types.add(p)

In [ ]:
#df[df['normed_av'] < .3][['responses', 'ans', 'gpt_score']]

In [ ]:
changes = []
for d in dataframes:
  d_list = []
  for p_l, c_l in zip(d['pos_list'], d['change_list']):

    ad_dict = {}
    for pos_type in all_types:
      ad_dict[pos_type] = 0

    for p, c in zip(p_l, c_l):
      ad_dict[p] += c
    d_list.append(ad_dict)
  type_changes = pd.DataFrame(d_list)
  type_changes = type_changes - type_changes.mean()
  type_changes['score'] = d[METRIC]
  type_changes['score'] = type_changes['score']  - .5
  score_corr = type_changes.corr()['score']
  type_change_list = type_changes.sum() * score_corr
  changes.append(type_change_list)
pos_score_sum1 = pd.DataFrame(changes).T
print(f"change estimate: {pos_score_sum1.drop('score').sum()}")
pos_score_sum1

In [ ]:
changes = []
for d in dataframes:
  d_list = []
  for p_l, c_l in zip(d['pos_list_lex'], d['change_list_lex']):

    ad_dict = {}
    for pos_type in all_types:
      ad_dict[pos_type] = 0

    for p, c in zip(p_l, c_l):
      ad_dict[p] += c
    d_list.append(ad_dict)
  type_changes = pd.DataFrame(d_list)
  type_changes = type_changes - type_changes.mean()

  type_changes['score'] = d[METRIC]
  type_changes['score'] = type_changes['score']  - .5
  score_corr = type_changes.corr()['score']
  type_change_list = type_changes.sum() * score_corr
  changes.append(type_change_list)
pos_score_sum2 = pd.DataFrame(changes).T
print(f"change estimate: {pos_score_sum2.drop('score').sum()}")
pos_score_sum2


In [ ]:
import copy
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time


class DeBERTaClassificationHead(nn.Module):
    """DeBERTa model with classification head."""
    def __init__(self, model_name, num_classes, dropout_rate=0.1):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        #pooled = outputs.last_hidden_state[:, 0, :]
        if torch.isnan(outputs).any():
            print("NaNs detected AFTER DeBERTa encoder!")
            print(f"inputs: {input_ids}")
            print(f"attention_mask: {attention_mask}")
            print("hidden state:")
            print(f"output shape: {outputs.shape}")
            print(f"outputs: {outputs}")

        pooled = outputs.mean(dim=1)
        if torch.isnan(pooled).any():
            print("NaNs detected AFTER pooling!")

            print("pooled stats:")
            print(f"min: {pooled.nanmin()}")
            print(f"max: {pooled.nanmax()}")

            nan_idx = torch.nonzero(torch.isnan(pooled))
            print("First pooled NaN index:", nan_idx[0])
        pooled = pooled.float()
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


def train_deberta_model(model, train_loader, val_loader, num_epochs=3, learning_rate=2e-5, device='cpu'):
    """Train a DeBERTa classification model."""
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()

    model = model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        start = time.perf_counter()
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # --- Additional checks for bad inputs / labels ---
            mask_sums = attention_mask.sum(dim=1)

            if torch.any(mask_sums == 0):
                print("Mask sum zero.")
                bad_rows = (mask_sums == 0)
                # Force CLS token visible
                attention_mask[bad_rows, 0] = 1
            if torch.any(input_ids < 0):
                print("Warning: Negative input_ids detected (possible NaN text).")
            if torch.any(attention_mask < 0):
                print("Warning: Negative attention_mask detected.")
            if torch.any(labels < 0):
                print("Warning: Negative labels detected (possible NaN in targets).")
            if hasattr(model, 'classifier') and hasattr(model.classifier, 'out_features'):
                if torch.any(labels >= model.classifier.out_features):
                    print(f"Warning: Labels out of bounds detected! Max label: {labels.max().item()}, Num classes: {model.classifier.out_features}")
            # -------------------------------------------------

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            if torch.isnan(loss):
                print(f"Warning: NaN loss detected!")
                print(f"labels {labels}")
                print(f"logits {logits}")
                print(f"input_ids {input_ids}")
                print(f"attention_mask {attention_mask}")
                # Optionally uncomment the line below to drop into the debugger
                # breakpoint()
            else:
              print("batch ok")

            loss.backward()

            # Add gradient clipping to prevent exploding gradients (common in DeBERTa)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            total_train_loss += loss.item()

        scheduler.step()
        print(f"Finished epoch {epoch+1}")
        end = time.perf_counter()
        print(f"Iteration {epoch+1} took {end - start:0.4f} seconds")
        avg_train_loss = total_train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        model.eval()
        total_val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                total_val_loss += loss.item()

                # Get predicted class indices (highest logit)
                preds = torch.argmax(logits, dim=-1)

                # Move to CPU and convert to list for sklearn metric evaluation
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)
        end = time.perf_counter()

        # --- METRIC CALCULATION ---
        # "macro" averaging works well for multi-class; change to "binary" if doing 2-class classification
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='macro', zero_division=0
        )
        accuracy = accuracy_score(all_labels, all_preds)

        # --- PERFORMANCE REPORT ---
        print(f"\n================ Epoch {epoch+1}/{num_epochs} ================")
        print(f"Time Elapsed   : {end - start:0.2f} seconds")
        print(f"Train Loss     : {avg_train_loss:.4f}")
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Accuracy       : {accuracy:.4f}")
        print(f"Precision (Mac): {precision:.4f}")
        print(f"Recall (Macro) : {recall:.4f}")
        print(f"F1-Score (Mac) : {f1:.4f}")
        print("=============================================")
    return model


In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
NUM_CLASSES = 3

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# =========================================================
# INSTANTIATE YOUR EXISTING CLASS
# =========================================================

model = DeBERTaClassificationHead(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES
).to(device)

model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =========================================================
# HELPER
# =========================================================

def run_forward_test(texts, description=""):

    print("\n")
    print("=" * 70)
    print(description)
    print("=" * 70)

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    print("input_ids shape:", input_ids.shape)
    print("attention_mask shape:", attention_mask.shape)

    print("input_ids min:", input_ids.min().item())
    print("input_ids max:", input_ids.max().item())

    print("vocab size:", model.deberta.config.vocab_size)

    print("attention sums:")
    print(attention_mask.sum(dim=1))

    print("NaNs in input_ids:",
          torch.isnan(input_ids.float()).any().item())

    print("NaNs in attention_mask:",
          torch.isnan(attention_mask.float()).any().item())

    # -----------------------------------------------------
    # EMBEDDINGS ONLY
    # -----------------------------------------------------

    with torch.no_grad():

        embeddings = model.deberta.embeddings(input_ids)

        print("\nEmbeddings shape:", embeddings.shape)

        print("NaNs in embeddings:",
              torch.isnan(embeddings).any().item())

        if torch.isnan(embeddings).any():

            nan_idx = torch.nonzero(torch.isnan(embeddings))

            print("First embedding NaN index:",
                  nan_idx[0])

            return

    # -----------------------------------------------------
    # FULL FORWARD PASS
    # -----------------------------------------------------

    with torch.no_grad():

        logits = model(input_ids, attention_mask)

    print("\nForward completed")

    if logits is None:
        print("Model returned None")
        return

    print("logits shape:", logits.shape)

    print("NaNs in logits:",
          torch.isnan(logits).any().item())

    if torch.isnan(logits).any():

        nan_idx = torch.nonzero(torch.isnan(logits))

        print("First logits NaN index:",
              nan_idx[0])

    print("\nlogits:")
    print(logits)


# =========================================================
# TEST 1 — NORMAL SENTENCES
# =========================================================

run_forward_test(
    [
        "This is a normal sentence.",
        "Testing DeBERTa forward pass.",
        "Nothing unusual here."
    ],
    "TEST 1 — NORMAL TEXT"
)

# =========================================================
# TEST 2 — SINGLE SHORT SAMPLE
# =========================================================

run_forward_test(
    [
        "hello world"
    ],
    "TEST 2 — SINGLE SAMPLE"
)

# =========================================================
# TEST 3 — EMPTY STRINGS
# =========================================================

run_forward_test(
    [
        "",
        " ",
        "   "
    ],
    "TEST 3 — EMPTY STRINGS"
)

# =========================================================
# TEST 4 — VERY LONG TEXT
# =========================================================

run_forward_test(
    [
        "test " * 2000
    ],
    "TEST 4 — LONG TEXT"
)

# =========================================================
# TEST 5 — RANDOM TOKEN IDS
# =========================================================

print("\n")
print("=" * 70)
print("TEST 5 — RANDOM TOKEN IDS")
print("=" * 70)

batch_size = 4
seq_len = 32

vocab_size = model.deberta.config.vocab_size

random_input_ids = torch.randint(
    0,
    vocab_size,
    (batch_size, seq_len),
    device=device
)

random_attention_mask = torch.ones(
    (batch_size, seq_len),
    dtype=torch.long,
    device=device
)

print("random_input_ids shape:",
      random_input_ids.shape)

print("random_input_ids min:",
      random_input_ids.min().item())

print("random_input_ids max:",
      random_input_ids.max().item())

with torch.no_grad():

    embeddings = model.deberta.embeddings(
        random_input_ids
    )

    print("\nNaNs in embeddings:",
          torch.isnan(embeddings).any().item())

    logits = model(
        random_input_ids,
        random_attention_mask
    )

print("\nNaNs in logits:",
      torch.isnan(logits).any().item())

print(logits)

In [ ]:
combined_texts = (df1['prompts'].astype(str) + " [SEP] " + df1["responses"].astype(str)).values

In [ ]:
combined_texts[5]

In [ ]:
run_forward_test(combined_texts[2:5])

In [ ]:

class TextClassificationDataset(Dataset):
    """Dataset for text classification with DeBERTa."""
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
df1[]

In [ ]:
df = df1
input_text_col = 'prompts'
output_text_col = 'responses'
metric_col = "correct"
position = (0,0,)
axes_names = ["knowledge", "reasoning"]
model_name = "microsoft/deberta-v3-small"
num_epochs=3
batch_size=8
learning_rate=2e-5
max_length=512
device=None
verbose=True

combined_texts = (df[input_text_col].astype(str) + " [SEP] " + df[output_text_col].astype(str)).values

# Generate targets
target_dict = {0: df[metric_col]}
deferral_options = {0: position}
for i, _ in enumerate(axes_names):
    pos = copy.deepcopy(position)
    pos = pos[:i] + (pos[i] + 1,) + pos[i+1:]
    deferral_options[i+1] = pos


target_dict[1] = df2[metric_col]
target_dict[2] = df3[metric_col]

target_df = pd.DataFrame(target_dict)
targets = target_df.apply(colab_llm_utils.misc.biased_idxmax, axis=1).values
#targets = target_df.idxmax(axis=1).values

all_classes = np.arange(len(axes_names) + 1)
n_classes = len(all_classes)
oof_preds = np.zeros((len(df), n_classes))

if True:
    print(f"target dict shape {target_df.shape}")
    print(f"targets: {pd.Series(targets).describe()}")
    print(f"Number of classes: {n_classes}")
    print(f"Device: {device}")

# Initialize tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    raise

# K-fold cross-validation
for fold_idx in range(5):
    if True:
        print(f"Training fold {fold_idx + 1}/{5}")

    train_mask = df['fold'] != fold_idx
    val_mask = df['fold'] == fold_idx

    X_train_texts = combined_texts[train_mask]
    X_val_texts = combined_texts[val_mask]
    y_train = targets[train_mask]
    y_val = targets[val_mask]

    # Create datasets
    train_dataset = TextClassificationDataset(
        X_train_texts, y_train, tokenizer, max_length=max_length
    )
    val_dataset = TextClassificationDataset(
        X_val_texts, y_val, tokenizer, max_length=max_length
    )

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)



In [ ]:
    # Initialize model for this fold
    model = DeBERTaClassificationHead(model_name, n_classes, dropout_rate=0.1)

    # Train model
    try:
        model = train_deberta_model(
            model, train_loader, val_loader,
            num_epochs=num_epochs,
            learning_rate=learning_rate,
            device=device
        )
    except Exception as e:
        print(f"Error during training on fold {fold_idx}: {e}")
        raise

    # Get predictions
    probs = predict_deberta_proba(model, val_loader, n_classes, device=device)

    # Align with all classes
    present_classes = np.arange(n_classes)
    aligned = np.zeros((len(X_val_texts), n_classes))
    aligned[:, present_classes] = probs

    oof_preds[val_mask] = aligned

    # Clean up to free memory
    del model, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()

In [ ]:
train_loader

In [ ]:
dir(train_loader)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate)

In [ ]:
for batch in train_loader:
            input_ids = batch['input_ids'].to("cuda")
            attention_mask = batch['attention_mask'].to("cuda")
            labels = batch['label'].to("cuda")
            logits = model(input_ids, attention_mask)
            print(logits)

            loss = criterion(logits, labels)

            if torch.isnan(loss):
                print(f"Warning: NaN loss detected!")
                print(f"labels {labels}")
                print(f"logits {logits}")
                print(f"input_ids {input_ids}")
                print(f"attention_mask {attention_mask}")
                # Optionally uncomment the line below to drop into the debugger
                # breakpoint()
            else:
              print("batch ok")

            loss.backward()
            optimizer.step()
            print(logits)

In [ ]:
logits = model(input_ids, attention_mask)

In [ ]:
logits